# Milestone 1: IPT + Bethe Lattice DMFT

Run the DMFT self-consistency loop with IPT solver on the Bethe lattice.
Verify:
- Convergence of the loop
- Three-peak spectral structure (quasiparticle peak + Hubbard bands)
- Quasiparticle weight Z between 0 and 1
- Self-energy tail Sigma(inf) = U/2 at half-filling

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
from dmft.config import DMFTParams
from dmft.solvers.ipt import IPTSolver
from dmft.dmft_loop import dmft_loop
from dmft.observables import spectral_function_from_poles, spectral_function_bethe
from dmft.plotting import plot_green_function, plot_self_energy, plot_convergence, plot_spectral_function

## 1. Run DMFT loop at U=2

In [ ]:
params = DMFTParams.half_filling(U=2.0, beta=50.0, n_matsubara=1024, M_g=2, M_h=2)
params.max_iter = 80
params.mix = 0.3

solver = IPTSolver()
result = dmft_loop(params, solver, verbose=True)

## 2. Convergence

In [ ]:
fig = plot_convergence(result['history'], f'IPT DMFT, U={params.U}')
plt.show()

## 3. Green's function and self-energy

In [ ]:
fig = plot_green_function(result['wn'], result['G_loc'], f'G_loc, U={params.U}, Z={result["Z"]:.3f}')
plt.show()

fig = plot_self_energy(result['wn'], result['Sigma'], U=params.U, title=f'Sigma, U={params.U}')
plt.show()

print(f'Z = {result["Z"]:.4f}')
print(f'n = {result["n_imp"]:.4f}')
print(f'Re Sigma(high freq) = {result["Sigma"][-1].real:.4f} (expect U/2 = {params.U/2:.1f})')

## 4. Spectral function A(omega)

Analytic continuation using the Bethe lattice formula on the real axis.

In [ ]:
omega = np.linspace(-4, 4, 2000)
broadening = 0.05

# Use pole representation for analytic continuation
poles = result['poles']
A_omega = spectral_function_from_poles(
    omega, params.mu, params.eps_d,
    poles.V, poles.eps, poles.W, poles.eta, poles.sigma_inf,
    broadening=broadening
)

fig = plot_spectral_function(omega, A_omega,
    title=f'Spectral function (IPT), U={params.U}, Z={result["Z"]:.3f}')
plt.show()

# Check sum rule: integral A(w) dw should be 1
dw = omega[1] - omega[0]
print(f'Sum rule: integral A(w)dw = {np.sum(A_omega)*dw:.4f} (expect 1.0)')

## 5. Compare different U values

In [ ]:
U_values = [1.0, 2.0, 3.0]
results_U = {}

for U in U_values:
    p = DMFTParams.half_filling(U=U, beta=50.0, n_matsubara=1024, M_g=2, M_h=2)
    p.max_iter = 80; p.mix = 0.3
    r = dmft_loop(p, IPTSolver(), verbose=False)
    results_U[U] = r
    print(f'U={U:.1f}: Z={r["Z"]:.4f}, n={r["n_imp"]:.4f}')

# Plot spectral functions
fig, ax = plt.subplots(figsize=(8, 5))
for U in U_values:
    r = results_U[U]
    p = r['poles']
    A = spectral_function_from_poles(omega, U/2, 0.0, p.V, p.eps, p.W, p.eta, p.sigma_inf, 0.05)
    ax.plot(omega, A, label=f'U={U:.1f}, Z={r["Z"]:.3f}')

ax.set_xlabel(r'$\omega$'); ax.set_ylabel(r'$A(\omega)$')
ax.legend(); ax.set_title('IPT spectral function evolution')
plt.tight_layout(); plt.show()